# ECE 214B Project 2: Colab Run All

This notebook runs the project from a lightweight Google Drive drop folder. It is a runner only: all experiment logic stays in `project/scripts/` and `project/src/`.

Expected Drive layout:

```text
MyDrive/
  214B_colab_drop/
    colab_run_all.ipynb
    project/
    data/                  # README only; raw data is not copied here by default
    outputs/
  path/to/S26_ECE_214B_Mini_Project_2.zip
```

The notebook extracts the dataset zip into local Colab storage before running experiments. Set Runtime > Change runtime type > GPU before running all cells.

In [ ]:
# Mount Google Drive, define paths, and extract dataset zip locally
from google.colab import drive
from pathlib import Path
import shutil
import zipfile

drive.mount("/content/drive")

DROP_DIR = "/content/drive/MyDrive/214B_colab_drop"
DATA_ZIP = "/content/drive/MyDrive/path/to/S26_ECE_214B_Mini_Project_2.zip"
EXTRACT_DIR = "/content/214B_data"

PROJECT_DIR = f"{DROP_DIR}/project"
DATA_DIR = f"{EXTRACT_DIR}/S26_ECE_214B_Mini_Project_2"
OUTPUT_ZIP = f"{DROP_DIR}/outputs/outputs_colab.zip"

drop_path = Path(DROP_DIR)
project_path = Path(PROJECT_DIR)
zip_path = Path(DATA_ZIP)
extract_path = Path(EXTRACT_DIR)

if not drop_path.is_dir():
    raise FileNotFoundError(f"DROP_DIR not found: {DROP_DIR}")
if not project_path.is_dir():
    raise FileNotFoundError(f"PROJECT_DIR not found: {PROJECT_DIR}")
for required in ["scripts", "src"]:
    if not (project_path / required).is_dir():
        raise FileNotFoundError(f"PROJECT_DIR is missing {required}/: {project_path}")
if not zip_path.is_file():
    raise FileNotFoundError(f"DATA_ZIP not found: {DATA_ZIP}")

if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)
print(f"Extracting {DATA_ZIP} -> {EXTRACT_DIR}")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_path)

def is_dataset_dir(path: Path) -> bool:
    return all((path / name).is_dir() for name in ["splits", "sessions", "wavs"])

candidate = Path(DATA_DIR)
if is_dataset_dir(candidate):
    detected_data_dir = candidate
elif is_dataset_dir(extract_path):
    detected_data_dir = extract_path
else:
    matches = [path for path in extract_path.rglob("*") if path.is_dir() and is_dataset_dir(path)]
    if not matches:
        raise FileNotFoundError("Could not find extracted dataset folder containing splits/, sessions/, and wavs/.")
    detected_data_dir = matches[0]

DATA_DIR = str(detected_data_dir)
Path(OUTPUT_ZIP).parent.mkdir(parents=True, exist_ok=True)

print(f"DROP_DIR: {DROP_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_ZIP: {DATA_ZIP}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_ZIP: {OUTPUT_ZIP}")

In [ ]:
# Runtime check
import platform

print(f"Python version: {platform.python_version()}")
try:
    import torch
    print(f"torch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU detected. Cheap experiments can run, but HuBERT/WavLM/RoBERTa will be slow or impractical.")
except ImportError:
    print("torch is not installed yet. The dependency install cell will install project requirements.")

In [ ]:
# Move into project and install dependencies
import os
import subprocess
import sys
from pathlib import Path

os.chdir(PROJECT_DIR)
print(f"cwd: {Path.cwd()}")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## Core Experiments

These include dataset audit, TF-IDF, demographic/simple descriptors, and lightweight acoustic descriptors.

In [ ]:
# Run core experiments
import subprocess
import sys

commands = [
    [sys.executable, "scripts/audit_dataset.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_descriptor_baselines.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_acoustic_descriptor_baseline.py", "--data_dir", DATA_DIR],
]
for command in commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)

## Transformer and Acoustic Embedding Experiments

These should be run on a Colab GPU runtime. If GPU is unavailable, the failure or slow runtime should be obvious.

In [ ]:
# Warn before transformer experiments if GPU is unavailable
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Transformer/audio embedding experiments may be very slow.")

In [ ]:
# HuBERT baseline
import subprocess
import sys

subprocess.run([sys.executable, "scripts/run_hubert_baseline.py", "--data_dir", DATA_DIR], check=True)

In [ ]:
# HuBERT layer sweep
import subprocess
import sys

layers = [str(layer) for layer in range(13)]
command = [sys.executable, "scripts/run_hubert_layer_sweep.py", "--data_dir", DATA_DIR, "--layers"] + layers
print("+", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# WavLM baseline
import subprocess
import sys

subprocess.run([sys.executable, "scripts/run_wavlm_baseline.py", "--data_dir", DATA_DIR], check=True)

In [ ]:
# DistilRoBERTa/RoBERTa frozen text embedding baseline
import subprocess
import sys

TEXT_EMBED_MODEL = "distilroberta-base"
subprocess.run([
    sys.executable,
    "scripts/run_roberta_text_baseline.py",
    "--data_dir",
    DATA_DIR,
    "--model-id",
    TEXT_EMBED_MODEL,
], check=True)

## Phase 2: Linguistic Marker Panel

Runs the clinically motivated, interpretable linguistic marker panel. This is still script-driven; experiment logic remains in `project/scripts/` and `project/src/`.


In [ ]:
# Phase 2 linguistic marker panel
import subprocess
import sys

subprocess.run([sys.executable, "scripts/run_linguistic_marker_panel.py", "--data_dir", DATA_DIR], check=True)


In [ ]:
# Phase 2 marker-based error analysis
import subprocess
import sys

subprocess.run([sys.executable, "scripts/analyze_marker_errors.py", "--data_dir", DATA_DIR], check=True)


## Phase 3: Uncertainty-Gated Marker Fusion

Use TF-IDF as the primary model and let linguistic marker evidence influence predictions only near the TF-IDF decision boundary. This remains analysis over saved model probabilities; it does not retrain or alter earlier outputs.


In [ ]:
# Phase 3 uncertainty-gated marker fusion
import subprocess
import sys

subprocess.run([sys.executable, "scripts/run_uncertainty_gated_marker_fusion.py"], check=True)


## Fusion, Thresholding, and Required Results

In [ ]:
# Run fusion, clinical thresholding, and required result collection
import subprocess
import sys

commands = [
    [sys.executable, "scripts/run_fusion.py"],
    [sys.executable, "scripts/run_clinical_thresholding.py"],
    [sys.executable, "scripts/run_age_aware_thresholding.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/collect_required_results.py"],
]
for command in commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)

## Future Phase 2/3 Placeholders

Do not implement or run these yet:

- Content coverage features
- Confidence-gated fusion
- Age-aware thresholding

In [ ]:
# Zip project outputs and reports back to the drop folder
import shutil
from pathlib import Path

output_zip_path = Path(OUTPUT_ZIP)
output_zip_path.parent.mkdir(parents=True, exist_ok=True)
package_dir = Path("_colab_package_outputs")
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir()
if Path("outputs").exists():
    shutil.copytree("outputs", package_dir / "outputs")
if Path("reports").exists():
    shutil.copytree("reports", package_dir / "reports")
archive_base = output_zip_path.with_suffix("")
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=package_dir)
shutil.rmtree(package_dir)
print(f"Wrote output zip: {zip_path}")